# AIMarx TRAIN-03 — Qwen3-0.6B guarded pilot to step 20

Free-tier T4 only. This notebook recreates the verified smoke checkpoint, resumes one effective epoch, evaluates validation, and downloads a ZIP. It does not use the smoke-test split, buy compute, mount Drive, or push a model.


In [ ]:
import os, subprocess, sys
assert os.path.exists('/content'), 'Run this notebook in Google Colab'
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
REPO = 'https://github.com/hongkhang21998-creator/AIMarx.git'
PINNED_COMMIT = 'ce188aca11c1bbaec8a0929e4172e66261ef2646'
subprocess.run(['git', 'clone', REPO, '/content/AIMarx'], check=True)
subprocess.run(['git', '-C', '/content/AIMarx', 'checkout', '--detach', PINNED_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', '/content/AIMarx', 'rev-parse', 'HEAD'], text=True).strip() == PINNED_COMMIT
os.chdir('/content/AIMarx')


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'training/colab_qwen06/requirements-colab.txt'], check=True)


## Recreate the verified step-5 resume checkpoint


In [ ]:
subprocess.run([sys.executable, '-m', 'training.colab_qwen06.prepare', '/content/aimarx-colab-data'], check=True)
subprocess.run([sys.executable, '-m', 'training.colab_qwen06.train', '--data', '/content/aimarx-colab-data', '--output', '/content/aimarx-colab-output', '--stop-after', '1'], check=True)
subprocess.run([sys.executable, '-m', 'training.colab_qwen06.train', '--data', '/content/aimarx-colab-data', '--output', '/content/aimarx-colab-output', '--resume-from', '/content/aimarx-colab-output/checkpoint-1'], check=True)


## Resume checkpoint 5 to step 20

Twenty optimizer steps with microbatch 1 and gradient accumulation 4 consume 80 training examples: one effective epoch.


In [ ]:
subprocess.run([sys.executable, '-m', 'training.colab_qwen06.pilot', '/content/aimarx-colab-data', '/content/aimarx-pilot-data', '/content/aimarx-colab-output/checkpoint-5'], check=True)
subprocess.run([sys.executable, '-m', 'training.colab_qwen06.train', '--data', '/content/aimarx-pilot-data', '--output', '/content/aimarx-pilot-output', '--resume-from', '/content/aimarx-colab-output/checkpoint-5'], check=True)


## Verify step 20 and compare validation loss

Lower validation loss is a technical signal only; it does not establish administrative quality.


In [ ]:
subprocess.run([sys.executable, '-m', 'training.colab_qwen06.evaluate', '--data', '/content/aimarx-pilot-data', '--checkpoint', '/content/aimarx-pilot-output/checkpoint-20', '--expected-step', '20', '--output', '/content/aimarx-pilot-output/evaluation-step20.json'], check=True)


In [ ]:
import hashlib, json, pathlib, shutil, zipfile
print(pathlib.Path('/content/aimarx-pilot-output/evaluation-step20.json').read_text())
archive = pathlib.Path(shutil.make_archive('/content/AIMarx-Qwen3-0.6B-LoRA-pilot-step20', 'zip', '/content/aimarx-pilot-output'))
h = hashlib.sha256()
with archive.open('rb') as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b''):
        h.update(chunk)
with zipfile.ZipFile(archive) as zf:
    names = zf.namelist()
    assert all(not pathlib.PurePosixPath(n).is_absolute() and '..' not in pathlib.PurePosixPath(n).parts for n in names)
print(json.dumps({'artifact': archive.name, 'bytes': archive.stat().st_size, 'sha256': h.hexdigest(), 'entries': len(names)}, indent=2))
from google.colab import files
files.download(str(archive))
